# Threads 貼文資料處理 (合併為單篇)

這個 Notebook 負責讀取 `範例.txt`，並將同一篇貼文內的所有串文合併在一起，同時清除掉原本的「(續 x / y)」等分段標籤，以「整篇貼文」為單位輸出 DataFrame。

In [ ]:
import re
import pandas as pd

def parse_threads_file_combined(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
        
    # 步驟一：用分隔線將主要貼文切開
    posts = content.split('------------------------------')
    
    structured_data = []
    
    for post in posts:
        post = post.strip()
        if not post:
            continue
            
        # 步驟二：萃取貼文編號
        post_num_match = re.search(r'【貼文 (\d+)】', post)
        if not post_num_match:
            continue
        post_num = post_num_match.group(1)
        
        # 步驟三：過濾 Meta Data (網址、內容：、作者、日期)
        content_start_match = re.search(r'內容：\n.*?\n\d{4}-\d{1,2}-\d{1,2}\n', post)
        if content_start_match:
            post_text = post[content_start_match.end():]
        else:
            post_text = post
            
        # 步驟四：清除對話串 (Threads) 的分段標記
        # 匹配類似:
        # （續 
        # 1
        # /
        # 3
        # 我們這次不切割，而是直接把它取代為空字串或換行，讓文章連貫
        marker_pattern = r'(?:（續\s*\n)?\s*\d+\s*\n\s*/\s*\n\s*\d+\s*'
        
        # 將標記替換為換行，避免段落黏在一起
        clean_text = re.sub(marker_pattern, '\n', post_text)
        
        # 步驟五：清除多餘的頭尾空白並組裝資料
        clean_text = clean_text.strip()
        
        structured_data.append({
            "貼文編號": f"貼文{post_num}",
            "文字內容": clean_text
        })
            
    return pd.DataFrame(structured_data)

# 執行函式並顯示結果
df_combined = parse_threads_file_combined('WhiteTalk.txt')
df_combined

,貼文編號,文字內容
0,貼文1,（吐槽文，不喜勿入）\n之前看到某個財經暢銷書的教授\n出來點出技術分析的爭議\n我也陸續發...
1,貼文2,最近有時間就來整理一些以前被問到的問題\n我發現剛接觸option的新人\n喜歡套 Blac...
2,貼文3,今天早上，一個來自對岸的友人問我怎麼看館長陳之漢去大陸「做交流」這件事\n老實說，我對這個人...
3,貼文4,利率市場引發的「錯誤定價」怎麼去交易？\n我記得當時2021年的環境\n台灣是個大牛\n很多...
4,貼文5,台灣貨幣升值，真的會必然引發熱錢流入、資產價格上漲嗎？\n網路上隨便一篇貼文開頭就說\n「央...
...,...,...
378,貼文388,巴菲特退休後，波克夏的挑戰才剛開始⋯⋯\n2025年巴菲特在波克夏股東大會上\n正式宣布年底...
379,貼文389,不少投資人喜歡操作「持個股 + 空指數期貨」的組合，\n表面上看來像是在幫自己的個股買保險，...
380,貼文390,有時候我們在宏觀上對利率的解讀\n對於做債的trader，解讀的視角可能跟我們差異甚大\n尤...
381,貼文391,有一次在對岸的一場非公開資產管理會議，\n一位前量化交易員分享過一句話，我非常認同：\n「在...


In [4]:
# 檢視資料是否符合預期，也可以匯出成 CSV 檔
df_combined.to_csv('structured_threads_by_post.csv', index=False, encoding='utf-8-sig')
df_combined.head()

,貼文編號,文字內容
0,貼文1,（吐槽文，不喜勿入）\n之前看到某個財經暢銷書的教授\n出來點出技術分析的爭議\n我也陸續發...
1,貼文2,最近有時間就來整理一些以前被問到的問題\n我發現剛接觸option的新人\n喜歡套 Blac...
2,貼文3,今天早上，一個來自對岸的友人問我怎麼看館長陳之漢去大陸「做交流」這件事\n老實說，我對這個人...
3,貼文4,利率市場引發的「錯誤定價」怎麼去交易？\n我記得當時2021年的環境\n台灣是個大牛\n很多...
4,貼文5,台灣貨幣升值，真的會必然引發熱錢流入、資產價格上漲嗎？\n網路上隨便一篇貼文開頭就說\n「央...
